# Урок 6. Кратчайший путь в графе

9 класс · I четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [← Урок 5](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-05.ipynb) · [Урок 7 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-07.ipynb)

---

Задача о кратчайшем пути. Перебор вариантов и последовательное уточнение. Классическое задание ОГЭ про дороги между пунктами.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 9А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="09-06", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Задача, которая встречается везде

Навигатор строит маршрут. Мессенджер ищет цепочку общих знакомых.
Логистическая компания планирует доставку. Во всех случаях решается
одна задача:

> Дан взвешенный граф. Найти путь между двумя вершинами
> с наименьшей суммой весов.

Обратите внимание: кратчайший путь — не обязательно тот, где меньше
рёбер. Прямая дорога может оказаться длиннее объезда через два города.

### Разбор на примере

```
           4            9
     А ─────────── Б ─────────── Г
       ╲           │           ╱ │
      7  ╲         │         ╱   │
           ╲     2 │       ╱ 3   │ 5
             ╲     │     ╱       │
               ╲   │   ╱         │
                   В ─────────── Д
                          6
```

Таблица тех же дорог:

|  | А | Б | В | Г | Д |
|---|---|---|---|---|---|
| **А** | — | 4 | 7 | — | — |
| **Б** | 4 | — | 2 | 9 | — |
| **В** | 7 | 2 | — | 3 | 6 |
| **Г** | — | 9 | 3 | — | 5 |
| **Д** | — | — | 6 | 5 | — |

Найдём кратчайший путь из А в Д. Переберём варианты:

| Путь | Длина |
|---|---|
| А → В → Д | 7 + 6 = 13 |
| А → Б → В → Д | 4 + 2 + 6 = **12** |
| А → Б → Г → Д | 4 + 9 + 5 = 18 |
| А → Б → В → Г → Д | 4 + 2 + 3 + 5 = 14 |
| А → В → Г → Д | 7 + 3 + 5 = 15 |
| А → В → Б → Г → Д | 7 + 2 + 9 + 5 = 23 |
| А → Б → Г → В → Д | 4 + 9 + 3 + 6 = 22 |

**Ответ: 12**, через Б и В. Заметьте, что прямая дорога А → В длиной 7
оказалась хуже, чем обход через Б длиной 4 + 2 = 6.

### Метод последовательного уточнения

Перебирать все пути годится, пока их мало. Есть способ надёжнее.
На той же идее построены настоящие алгоритмы поиска кратчайших путей —
например, алгоритм Дейкстры, который вы разберёте в 11 классе.

Идея: для каждой вершины храним **лучшее известное расстояние** от старта.
Сначала все расстояния бесконечны, кроме старта — там ноль.
Затем многократно проходим по всем рёбрам и улучшаем оценки.

```
  Шаг 0:  А=0   Б=∞   В=∞   Г=∞   Д=∞

  Из А:   Б = 0+4 = 4       В = 0+7 = 7
  Шаг 1:  А=0   Б=4   В=7   Г=∞   Д=∞

  Из Б:   В = 4+2 = 6 < 7 — улучшили!   Г = 4+9 = 13
  Шаг 2:  А=0   Б=4   В=6   Г=13  Д=∞

  Из В:   Г = 6+3 = 9 < 13 — улучшили!  Д = 6+6 = 12
  Шаг 3:  А=0   Б=4   В=6   Г=9   Д=12

  Из Г:   Д = 9+5 = 14 — хуже, чем 12, не улучшаем
  Итог:   Д = 12
```

Правило одно: если через текущую вершину до соседа получается **короче**,
чем известно сейчас, — записываем новое расстояние. Такая операция
называется **релаксацией ребра**.

### Когда останавливаться

Проходов нужно столько, пока хоть что-то улучшается. Для графа
из n вершин достаточно n−1 проходов — дальше улучшать нечего.

### Что важно на экзамене

В задачах ОГЭ граф маленький: 5–7 вершин. Перебор путей вручную вполне
реален, и обычно он быстрее. Но проверяйте себя: самая частая ошибка —
**пропустить вариант**. Выписывайте пути системно, а не как вспомнится:
сначала все пути через первого соседа, потом через второго.

## Смотрим, как это работает

### Пример 1. Перебор всех путей

Найдём все пути и их длины — так, как вы делали бы на бумаге,
но ничего не пропустив.

In [ ]:
пункты = ["А", "Б", "В", "Г", "Д"]
веса = [
    [0, 4, 7, 0, 0],
    [4, 0, 2, 9, 0],
    [7, 2, 0, 3, 6],
    [0, 9, 3, 0, 5],
    [0, 0, 6, 5, 0],
]


def все_пути(текущая, финиш, посещённые, длина, путь):
    if текущая == финиш:
        print(f"  {' → '.join(путь)}  длина {длина}")
        return

    for сосед in range(len(пункты)):
        вес = веса[текущая][сосед]
        if вес != 0 and сосед not in посещённые:
            все_пути(сосед, финиш, посещённые | {сосед}, длина + вес,
                     путь + [пункты[сосед]])


print("Все пути из А в Д:")
все_пути(0, 4, {0}, 0, ["А"])

Функция вызывает саму себя — это **рекурсия**, и мы разберём её подробно
на уроке 10. Пока достаточно понять идею: из текущей вершины пробуем
пойти в каждого непосещённого соседа и продолжаем поиск оттуда.

Множество `посещённые` не даёт ходить по кругу, а запись
`посещённые | {сосед}` создаёт новое множество с добавленной вершиной,
не портя старое.

### Пример 2. Метод последовательного уточнения

Реализуем приём с таблицей расстояний.

In [ ]:
БЕСКОНЕЧНОСТЬ = float("inf")


def кратчайшие(веса, старт):
    n = len(веса)
    расстояния = [БЕСКОНЕЧНОСТЬ] * n
    расстояния[старт] = 0

    for проход in range(n - 1):
        улучшили = False
        for из_ in range(n):
            for в in range(n):
                вес = веса[из_][в]
                if вес != 0 and расстояния[из_] + вес < расстояния[в]:
                    расстояния[в] = расстояния[из_] + вес
                    улучшили = True
        print(f"  после прохода {проход + 1}: {расстояния}")
        if not улучшили:
            break

    return расстояния


итог = кратчайшие(веса, 0)
print()
for i, пункт in enumerate(пункты):
    print(f"  А → {пункт}: {итог[i]}")

Значение `float("inf")` — настоящая бесконечность в Python: она больше
любого числа, поэтому первая же найденная дорога её улучшит.
Использовать вместо неё большое число вроде 999999 можно, но некрасиво:
а вдруг реальный путь окажется длиннее?

Флаг `улучшили` позволяет остановиться раньше, если за целый проход
ничего не изменилось — значит, лучше уже не будет.

### Пример 3. Восстановление самого пути

Знать длину — половина дела. Часто нужен и сам маршрут.
Для этого запоминаем, откуда мы пришли в каждую вершину.

In [ ]:
def путь_с_маршрутом(веса, старт, финиш):
    n = len(веса)
    расстояния = [БЕСКОНЕЧНОСТЬ] * n
    откуда = [None] * n
    расстояния[старт] = 0

    for _ in range(n - 1):
        for из_ in range(n):
            for в in range(n):
                вес = веса[из_][в]
                if вес != 0 and расстояния[из_] + вес < расстояния[в]:
                    расстояния[в] = расстояния[из_] + вес
                    откуда[в] = из_

    # разматываем маршрут с конца
    маршрут = []
    текущая = финиш
    while текущая is not None:
        маршрут.append(пункты[текущая])
        текущая = откуда[текущая]

    return расстояния[финиш], маршрут[::-1]


длина, маршрут = путь_с_маршрутом(веса, 0, 4)
print(f"Кратчайший путь из А в Д: {' → '.join(маршрут)}, длина {длина}")

Массив `откуда` хранит предшественника каждой вершины. Пройдя по нему
от финиша назад, получаем маршрут в обратном порядке — а срез `[::-1]`
его разворачивает.

## Пробуем сами

### Задача 1. Длина заданного пути

По весовой матрице и списку номеров вершин верните суммарную длину
такого маршрута. Если между какими-то соседними вершинами дороги нет —
верните `-1`.

In [ ]:
def длина_пути(веса, маршрут):
    return ...

In [ ]:
si.check("1", длина_пути, [
    (([[0, 4, 7, 0, 0], [4, 0, 2, 9, 0], [7, 2, 0, 3, 6],
       [0, 9, 3, 0, 5], [0, 0, 6, 5, 0]], [0, 1, 2, 4]), 12),
    (([[0, 4, 7, 0, 0], [4, 0, 2, 9, 0], [7, 2, 0, 3, 6],
       [0, 9, 3, 0, 5], [0, 0, 6, 5, 0]], [0, 2, 4]), 13),
    (([[0, 4], [4, 0]], [0, 1]), 4),
    (([[0, 0], [0, 0]], [0, 1]), -1),
])

### Задача 2. Кратчайшее расстояние

Реализуйте метод последовательного уточнения: по весовой матрице,
номеру старта и номеру финиша верните длину кратчайшего пути.
Если пути нет — верните `-1`.

Опирайтесь на пример 2.

In [ ]:
def кратчайшее(веса, старт, финиш):
    return ...

In [ ]:
si.check("2", кратчайшее, [
    (([[0, 4, 7, 0, 0], [4, 0, 2, 9, 0], [7, 2, 0, 3, 6],
       [0, 9, 3, 0, 5], [0, 0, 6, 5, 0]], 0, 4), 12),
    (([[0, 4, 7, 0, 0], [4, 0, 2, 9, 0], [7, 2, 0, 3, 6],
       [0, 9, 3, 0, 5], [0, 0, 6, 5, 0]], 0, 3), 9),
    (([[0, 1], [1, 0]], 0, 1), 1),
    (([[0, 0], [0, 0]], 0, 1), -1),
])

### Задача 3. Проверка на бумаге

По той же таблице дорог найдите кратчайший путь **из Б в Д**.
Считайте вручную, перебирая варианты, и впишите длину числом.

|  | А | Б | В | Г | Д |
|---|---|---|---|---|---|
| **А** | — | 4 | 7 | — | — |
| **Б** | 4 | — | 2 | 9 | — |
| **В** | 7 | 2 | — | 3 | 6 |
| **Г** | — | 9 | 3 | — | 5 |
| **Д** | — | — | 6 | 5 | — |

In [ ]:
ответ = 0

si.check_value("3", ответ, "2c624232cdd22177",
               hint="Из Б в В всего 2, а из В в Д — 6. Есть ли короче?")

## Домашнее задание

### Домашнее задание 1. Все расстояния от вершины

Верните список кратчайших расстояний от заданной вершины до всех
остальных. Недостижимые вершины обозначьте числом `-1`.

In [ ]:
def все_расстояния(веса, старт):
    return ...

In [ ]:
si.check("дз1", все_расстояния, [
    (([[0, 4, 7, 0, 0], [4, 0, 2, 9, 0], [7, 2, 0, 3, 6],
       [0, 9, 3, 0, 5], [0, 0, 6, 5, 0]], 0), [0, 4, 6, 9, 12]),
    (([[0, 1], [1, 0]], 0), [0, 1]),
    (([[0, 0], [0, 0]], 0), [0, -1]),
])

### Домашнее задание 2. Самая удалённая вершина

Верните номер вершины, до которой от старта идти дальше всего.
Недостижимые вершины не учитывайте. Если таких несколько —
верните наименьший номер.

In [ ]:
def самая_дальняя(веса, старт):
    return ...

In [ ]:
si.check("дз2", самая_дальняя, [
    (([[0, 4, 7, 0, 0], [4, 0, 2, 9, 0], [7, 2, 0, 3, 6],
       [0, 9, 3, 0, 5], [0, 0, 6, 5, 0]], 0), 4),
    (([[0, 1], [1, 0]], 0), 1),
    (([[0, 0], [0, 0]], 0), 0),
])

### Домашнее задание 3. Маршрут, а не только длина

Верните сам кратчайший маршрут — список номеров вершин от старта
до финиша. Если пути нет — верните пустой список.

Опирайтесь на пример 3: заведите массив `откуда` и разматывайте
маршрут с конца.

In [ ]:
def маршрут(веса, старт, финиш):
    return ...

In [ ]:
si.check("дз3", маршрут, [
    (([[0, 4, 7, 0, 0], [4, 0, 2, 9, 0], [7, 2, 0, 3, 6],
       [0, 9, 3, 0, 5], [0, 0, 6, 5, 0]], 0, 4), [0, 1, 2, 4]),
    (([[0, 1], [1, 0]], 0, 1), [0, 1]),
    (([[0, 0], [0, 0]], 0, 1), []),
])

---

### Тренировка к экзамену

Возьмите таблицу из урока и найдите кратчайшие пути между всеми парами
пунктов вручную. Их всего десять. Затем проверьте себя функцией
`все_расстояния` — она посчитает то же самое за долю секунды.

Умение делать это на бумаге за минуту и есть то, что проверяет экзамен.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 5](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-05.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 7 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-09/urok-07.ipynb)